In [1]:
# Подключение библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import datetime as dt
import os

In [2]:
# Входные данные
file_year = 2014 # Год файла
interval = 15 # 15 минутный интервал вокруг грозового разряда и вокруг момента совпадения

thunderbolts_npz_in = f'C:/Users/Maks/Desktop/Jupyter/thunderbolts_data/npz/{file_year}_thunderbolts_clastered.npz'
thunderbolts_hdf_out = f'C:/Users/Maks/Desktop/Jupyter/thunderbolts_data/hdf/{file_year}_thunderbolts_clastered.h5'
satellite_directory = f'C:/Users/Maks/Desktop/Jupyter/satellite_data/TIDI/{file_year}'
output_filtered_satellite_dir = f'C:/Users/Maks/Desktop/Jupyter/output_filtered_satellite_data/{file_year}'
output_full_data_dir = "C:/Users/Maks/Desktop/Jupyter/output_full_data"

In [3]:
# Преобразование из формата .npz в .hdf, удаление лишних данных, установление даты в качестве индекса
data = np.load(thunderbolts_npz_in)
data_npz = pd.DataFrame(data['strikes']).drop('tail', axis=1).set_index('date')
data_npz.to_hdf(thunderbolts_hdf_out, key='strikes', mode='w', complevel=9)
data_hdf = pd.read_hdf(thunderbolts_hdf_out, 'strikes')

In [ ]:
# Выявление файлов .vec с некорректными данными (Data_OK=b'F')
files_to_delete = []
for filename in os.listdir(satellite_directory):
    file_path = os.path.join(satellite_directory, filename)
    satellite_data = xr.open_dataset(file_path, decode_timedelta=True)
    if (satellite_data['data_ok'].values == b'F').any():
        files_to_delete.append(file_path)
        print(f"Файл {filename} содержит data_ok=b'F'")
    satellite_data.close()
if len(files_to_delete) == 0:
    print("Файлов с data_ok=b'F' не найдено")

In [35]:
# Функция удаления отдаленных разрядов и выделения области наибольшего скопления
def simple_remove_outliers(group, percentile=90, plot=False, cluster_id=None):
    if len(group) <= 5:
        # Для малых кластеров находим самую плотную группу разрядов
        if len(group) == 1:
            # Для одного разряда - минимальная область вокруг него
            lat, lon = group['lat'].iloc[0], group['lon'].iloc[0]
            fixed_span = 0.09  # 10×10 км
            bounds = (
                lat - fixed_span/2,
                lat + fixed_span/2,
                lon - fixed_span/2,
                lon + fixed_span/2
            )
        else:
            # Для 2-5 разрядов находим пару самых близких точек
            coords = group[['lat', 'lon']].values
            min_distance = float('inf')
            closest_pair = None
            # Находим две самые близкие точки
            for i in range(len(coords)):
                for j in range(i+1, len(coords)):
                    distance = np.sqrt((coords[i][0] - coords[j][0])**2 + (coords[i][1] - coords[j][1])**2)
                    if distance < min_distance:
                        min_distance = distance
                        closest_pair = (coords[i], coords[j])
            if closest_pair:
                # Центр между двумя самыми близкими точками
                lat_center = (closest_pair[0][0] + closest_pair[1][0]) / 2
                lon_center = (closest_pair[0][1] + closest_pair[1][1]) / 2
                # Определяем размер области на основе распределения точек
                distances_from_center = np.sqrt(
                    (group['lat'] - lat_center)**2 + (group['lon'] - lon_center)**2
                )
                # Берем 75% перцентиль расстояний + минимальный размер
                radius = max(np.percentile(distances_from_center, 75), 0.045)  # минимум 5 км
                bounds = (
                    lat_center - radius,
                    lat_center + radius,
                    lon_center - radius,
                    lon_center + radius
                )
            else:
                # Запасной вариант
                lat_center = group['lat'].mean()
                lon_center = group['lon'].mean()
                fixed_span = 0.09
                bounds = (
                    lat_center - fixed_span/2,
                    lat_center + fixed_span/2,
                    lon_center - fixed_span/2,
                    lon_center + fixed_span/2
                )
    else:
        # Для больших кластеров - обычная фильтрация по перцентилям
        cut_percent = (100 - percentile) / 2
        lat_min = np.percentile(group['lat'], cut_percent)
        lat_max = np.percentile(group['lat'], 100 - cut_percent)
        lon_min = np.percentile(group['lon'], cut_percent)
        lon_max = np.percentile(group['lon'], 100 - cut_percent)
        bounds = (lat_min, lat_max, lon_min, lon_max)
    if plot:
        plot_cluster_comparison(group, bounds, cluster_id, 
                               "Dense cluster" if len(group) <= 5 else f"Filtered (percentile={percentile}%)")
    return bounds

In [36]:
# Алгоритм по поиску совпадений между пролетом спутника и областью грозового кластера
results = []
files_processed = 0
files_error = []

for filename in os.listdir(satellite_directory):
    file_path = os.path.join(satellite_directory, filename)

    try:
        with xr.open_dataset(file_path, decode_timedelta=True, engine='netcdf4') as satellite_data:
            # Проверяем наличие необходимых переменных
            if 'time' not in satellite_data.variables or 'lat' not in satellite_data.variables or 'lon' not in satellite_data.variables:
                print(f"Файл {filename} не содержит необходимых переменных (time, lat, lon)")
                files_error.append(filename)
                continue
    
            dataframe_satellite = pd.DataFrame({'date': satellite_data['time'], 'lat': satellite_data['lat'], 'lon': satellite_data['lon']})
            dataframe_satellite['date'] += dt.datetime(1980,1,6,0,0,0)
            dataframe_satellite = dataframe_satellite.set_index('date')
            
            # Вычисляем временной интервал для файла
            year = int(filename[8:12])
            day_of_year = int(filename[12:15])
            time_start = dt.datetime(year, 1, 1) + dt.timedelta(day_of_year - 1)
            time_end = time_start + dt.timedelta(1)
            
            # Фильтрация файла с данными о грозах data_hdf
            mask = (data_hdf.index >= time_start) & (data_hdf.index <= time_end)
            filtered_hdf = data_hdf[mask]
            
            if not filtered_hdf.empty:
                # Обрабатываем только кластеры с clnb > 0
                clusters = filtered_hdf[filtered_hdf['clnb'] > 0].groupby('clnb')
                
                for clnb, group in clusters:
                    # Вычисляем границы, без учета отдаленных грозовых разрядов, в зоне скопления
                    lat_min, lat_max, lon_min, lon_max = simple_remove_outliers(
                        group, percentile=90, plot=False
                    )
                    
                    # Проверяем каждый грозовой разряд в кластере
                    for flash_time in group.index:
                        # Временное окно 15 минут после грозового разряда
                        time_min = flash_time
                        time_max = flash_time + dt.timedelta(minutes=interval)
                        
                        # Поиск совпадений в спутниковых данных
                        mask = (
                            (dataframe_satellite.index >= time_min) & 
                            (dataframe_satellite.index <= time_max) &
                            (dataframe_satellite['lat'] >= lat_min) & 
                            (dataframe_satellite['lat'] <= lat_max) & 
                            (dataframe_satellite['lon'] >= lon_min) & 
                            (dataframe_satellite['lon'] <= lon_max)
                        )
                        matched_data = dataframe_satellite[mask]
                        
                        # Обнаружение и обработка совпадений
                        if dataframe_satellite[mask].any().any():
                            matched_data = dataframe_satellite[mask]
                            first_match = matched_data.iloc[0]
                            match_time = dataframe_satellite[mask].index[0]
                            
                            # Находим все разряды в кластере до момента совпадения (включая совпавший)
                            flashes_before_match = group[group.index <= match_time]
                            
                            if len(flashes_before_match) > 0:
                                last_flash_before = flashes_before_match.index.max()  # совпавший разряд
                                
                                # Находим разряды ЗА 15 МИН ДО совпавшего (исключая сам совпавший разряд)
                                time_15min_before = last_flash_before - dt.timedelta(minutes=interval)
                                flashes_in_15min_before = group[
                                    (group.index >= time_15min_before) & 
                                    (group.index < last_flash_before)  # строго меньше, исключаем совпавший
                                ]
                                
                                flash_times_before = flashes_in_15min_before.index.tolist()
                                
                                # Амплитуда совпавшего разряда 
                                flash_amp = group.loc[last_flash_before, 'amp']
                                
                                # Амплитуды разрядов за 15 мин ДО совпавшего
                                flash_amps_before = [row['amp'] for idx, row in flashes_in_15min_before.iterrows()]
                                
                            else:
                                last_flash_before = []
                                flash_times_before = []
                                flash_amps_before = []
                                flash_amp = []
        
                            results.append({
                                'clnb': clnb,
                                'satellite_file_name': filename,
                                'matched_time': match_time,  # Время пролета спутника
                                'flash_time': last_flash_before,  # Время грозового разряда перед пролетом
                                'time_dif': match_time - last_flash_before,
                                'flash_amp': flash_amp,
                                'flash_times_before': flash_times_before,  # только за 15 мин до
                                'flash_amps_before': flash_amps_before,    # только за 15 мин до
                                # Координаты
                                'lat_sat': first_match['lat'],   
                                'lon_sat': first_match['lon'],   
                                'lat_min': lat_min,
                                'lat_max': lat_max,
                                'lon_min': lon_min,
                                'lon_max': lon_max,
                            })
                            # Сохраняем выделенные данные пролета спутника в HDF
                            output_filename = 'trimmed_' + filename[0:-4] + '_clnb_' + str(clnb) + '.h5'
                            os.makedirs(output_filtered_satellite_dir, exist_ok=True) 
                            output_path = os.path.join(output_filtered_satellite_dir, output_filename)
                            # Фильтруем данные за ±15 минут вокруг момента совпадения
                            trim_time_min = matched_data.index[0] - dt.timedelta(minutes=interval)
                            trim_time_max = matched_data.index[0] + dt.timedelta(minutes=interval)
                            trim_mask = (
                                (dataframe_satellite.index >= trim_time_min) & 
                                (dataframe_satellite.index <= trim_time_max)
                            )
                            trimmed_data = dataframe_satellite[trim_mask]
                            # Удаление лишнего интервала пролета в файле (с одинаковым временем пролета, но другими координатами)
                            trimmed_data_reset = trimmed_data.reset_index()
                            # Находим строку совпадения по времени и координатам 
                            match_condition = (
                                (trimmed_data_reset['date'] == matched_data.index[0]) & 
                                (trimmed_data_reset['lat'] == first_match['lat']) & 
                                (trimmed_data_reset['lon'] == first_match['lon'])
                            )
                            match_idx_num = trimmed_data_reset[match_condition].index[0]
                            # Ищем разрыв ВВЕРХ (от точки совпадения к более поздним временам)
                            keep_indices_after = []
                            current_time = trimmed_data_reset.iloc[match_idx_num]['date']
                            for i in range(match_idx_num, len(trimmed_data_reset)):
                                if trimmed_data_reset.iloc[i]['date'] >= current_time:
                                    keep_indices_after.append(i)
                                    current_time = trimmed_data_reset.iloc[i]['date']
                                else:
                                    break
                            # Ищем разрыв ВНИЗ (от точки совпадения к более ранним временам)  
                            keep_indices_before = []
                            current_time = trimmed_data_reset.iloc[match_idx_num]['date']
                            for i in range(match_idx_num, -1, -1):
                                if trimmed_data_reset.iloc[i]['date'] <= current_time:
                                    keep_indices_before.append(i)
                                    current_time = trimmed_data_reset.iloc[i]['date']
                                else:
                                    break
                            # Объединяем индексы и берем только уникальные
                            keep_indices = list(set(keep_indices_before + keep_indices_after))
                            keep_indices.sort()
                            # Создаем отфильтрованные данные 
                            trimmed_data_filtered = trimmed_data_reset.iloc[keep_indices].set_index('date')
                            # Сохраняем в HDF
                            trimmed_data_filtered.to_hdf(output_path, key='satellite_data', mode='w')
                            break  # Прерываем после первого совпадения для этого кластера
                            
    except Exception as e:
            print(f"Ошибка при обработке файла {filename}: {e}")
            files_error.append(filename)
            continue
        
    files_processed += 1
    if files_processed % 10 == 0:  # Прогресс каждые 10 файлов
        print(f"Обработано файлов: {files_processed}")
            
# Создание итогового DataFrame
if results:
    satellite_overpass_matching = pd.DataFrame(results).set_index('clnb').sort_index()
    
    # Сохраняем результат в HDF файл
    os.makedirs(output_full_data_dir, exist_ok=True)
    output_hdf_path = os.path.join(output_full_data_dir, f'{file_year}_satellite_overpass_matching.h5')
    satellite_overpass_matching.to_hdf(output_hdf_path, key='matches', mode='w', complevel=9)

else:
    satellite_overpass_matching = pd.DataFrame(columns=['satellite_file_name', 'flash_time', 'matched_time'])
    print("Совпадений не найдено")
    
# Вывод статистики
print(f"\nРезультаты сопоставления:")
print(f"Обработано файлов: {len(os.listdir(satellite_directory))}")
print(f"Найдено совпадений: {len(satellite_overpass_matching)}")
print(f"Файлов с ошибками: {len(files_error)}")
if files_error:
    print(f"Проблемные файлы: {files_error[:5]}")  # Покажем первые 5

satellite_overpass_matching.T

Обработано файлов: 10
Обработано файлов: 20
Обработано файлов: 30
Обработано файлов: 40
Обработано файлов: 50
Обработано файлов: 60
Обработано файлов: 70
Обработано файлов: 80
Ошибка при обработке файла TIDI_PB_2014041_P0100_S0450_D011_R02.VEC: zero-size array to reduction operation fmin which has no identity
Обработано файлов: 90
Обработано файлов: 100
Обработано файлов: 110
Обработано файлов: 120
Обработано файлов: 130
Обработано файлов: 140
Обработано файлов: 150
Обработано файлов: 160
Обработано файлов: 170
Обработано файлов: 180
Обработано файлов: 190
Обработано файлов: 200
Обработано файлов: 210
Обработано файлов: 220
Обработано файлов: 230
Обработано файлов: 240
Обработано файлов: 250
Обработано файлов: 260
Обработано файлов: 270
Обработано файлов: 280
Обработано файлов: 290
Обработано файлов: 300
Обработано файлов: 310
Обработано файлов: 320
Обработано файлов: 330
Обработано файлов: 340
Обработано файлов: 350
Обработано файлов: 360
Обработано файлов: 370
Обработано файлов: 380


C:\Temp\ipykernel_17080\3074587509.py:174: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['satellite_file_name', 'flash_amp', 'flash_times_before',
       'flash_amps_before'],
      dtype='object')]

  satellite_overpass_matching.to_hdf(output_hdf_path, key='matches', mode='w', complevel=9)


clnb,155,375,637,637,750,818,914,922,1008,1008,...,15026,15099,15154,15163,15164,15174,15191,15194,15383,15552
satellite_file_name,TIDI_PB_2014122_P0100_S0450_D011_R01.VEC,TIDI_PB_2014133_P0100_S0450_D011_R01.VEC,TIDI_PB_2014139_P0100_S0450_D011_R01.VEC,TIDI_PB_2014139_P0100_S0450_D011_R02.VEC,TIDI_PB_2014140_P0100_S0450_D011_R01.VEC,TIDI_PB_2014141_P0100_S0450_D011_R02.VEC,TIDI_PB_2014142_P0100_S0450_D011_R01.VEC,TIDI_PB_2014142_P0100_S0450_D011_R01.VEC,TIDI_PB_2014144_P0100_S0450_D011_R01.VEC,TIDI_PB_2014144_P0100_S0450_D011_R02.VEC,...,TIDI_PB_2014241_P0100_S0450_D011_R01.VEC,TIDI_PB_2014242_P0100_S0450_D011_R01.VEC,TIDI_PB_2014243_P0100_S0450_D011_R01.VEC,TIDI_PB_2014243_P0100_S0450_D011_R01.VEC,TIDI_PB_2014243_P0100_S0450_D011_R01.VEC,TIDI_PB_2014243_P0100_S0450_D011_R01.VEC,TIDI_PB_2014243_P0100_S0450_D011_R01.VEC,TIDI_PB_2014243_P0100_S0450_D011_R01.VEC,TIDI_PB_2014247_P0100_S0450_D011_R01.VEC,TIDI_PB_2014253_P0100_S0450_D011_R01.VEC
matched_time,2014-05-02 16:02:42,2014-05-13 13:42:14,2014-05-19 20:11:07,2014-05-19 20:11:07,2014-05-20 20:20:38,2014-05-21 18:57:34,2014-05-22 19:11:23,2014-05-22 19:11:23,2014-05-24 08:07:41,2014-05-24 08:07:41,...,2014-08-29 15:54:37,2014-08-30 14:27:06,2014-08-31 04:55:00,2014-08-31 08:10:59,2014-08-31 08:10:59,2014-08-31 09:52:12,2014-08-31 11:33:26,2014-08-31 13:14:38,2014-09-04 07:26:43,2014-09-10 02:19:01
flash_time,2014-05-02 16:02:33,2014-05-13 13:40:21,2014-05-19 20:10:05,2014-05-19 20:10:05,2014-05-20 20:19:39,2014-05-21 18:57:31,2014-05-22 19:04:13,2014-05-22 19:10:53,2014-05-24 08:06:35,2014-05-24 08:06:35,...,2014-08-29 15:54:33,2014-08-30 14:24:30,2014-08-31 04:52:08,2014-08-31 08:08:45,2014-08-31 08:09:51,2014-08-31 09:51:47,2014-08-31 11:33:17,2014-08-31 13:13:25,2014-09-04 07:21:24,2014-09-10 02:16:00
time_dif,0 days 00:00:09,0 days 00:01:53,0 days 00:01:02,0 days 00:01:02,0 days 00:00:59,0 days 00:00:03,0 days 00:07:10,0 days 00:00:30,0 days 00:01:06,0 days 00:01:06,...,0 days 00:00:04,0 days 00:02:36,0 days 00:02:52,0 days 00:02:14,0 days 00:01:08,0 days 00:00:25,0 days 00:00:09,0 days 00:01:13,0 days 00:05:19,0 days 00:03:01
flash_amp,-16977.0,10669.0,10946.0,10946.0,-4645.0,17067.0,-3669.0,10383.0,4943.0,4943.0,...,10508.0,-13171.0,-3967.0,3081.0,5378.0,3408.0,2451.0,date 2014-08-31 13:13:25 -32000.0 2014-08-31...,-23442.0,-738.0
flash_times_before,"[2014-05-02 15:47:52, 2014-05-02 15:48:01, 201...",[2014-05-13 13:39:12],"[2014-05-19 19:56:50, 2014-05-19 19:57:20, 201...","[2014-05-19 19:56:50, 2014-05-19 19:57:20, 201...","[2014-05-20 20:05:36, 2014-05-20 20:07:49, 201...","[2014-05-21 18:42:35, 2014-05-21 18:42:39, 201...","[2014-05-22 18:54:42, 2014-05-22 18:59:49]","[2014-05-22 18:55:57, 2014-05-22 18:55:57, 201...","[2014-05-24 08:04:16, 2014-05-24 08:04:16, 201...","[2014-05-24 08:04:16, 2014-05-24 08:04:16, 201...",...,"[2014-08-29 15:40:11, 2014-08-29 15:40:24, 201...","[2014-08-30 14:11:46, 2014-08-30 14:13:33, 201...","[2014-08-31 04:37:32, 2014-08-31 04:38:44, 201...",[2014-08-31 07:57:11],"[2014-08-31 07:55:01, 2014-08-31 07:55:27, 201...","[2014-08-31 09:37:44, 2014-08-31 09:38:08, 201...","[2014-08-31 11:18:20, 2014-08-31 11:18:46, 201...","[2014-08-31 12:58:44, 2014-08-31 12:59:11, 201...","[2014-09-04 07:08:10, 2014-09-04 07:14:39, 201...","[2014-09-10 02:07:48, 2014-09-10 02:09:54]"
flash_amps_before,"[-16719.0, 9107.0, -9630.0, -11363.0, 6504.0, ...",[-3022.0],"[-11866.0, -22540.0, -32000.0, -18293.0, 14012...","[-11866.0, -22540.0, -32000.0, -18293.0, 14012...","[4869.0, -3206.0, -2294.0, -1438.0, -4383.0, -...","[18067.0, 10300.0, -32000.0, 32000.0, 24286.0,...","[-1232.0, 5834.0]","[3327.0, 4870.0, 14497.0, 9210.0, 4961.0, 5475...","[20906.0, 2557.0, 2943.0]","[20906.0, 2557.0, 2943.0]",...,"[-9225.0, -8764.0, 6266.0, 24615.0, 23082.0, -...","[-20042.0, 13958.0, 19920.0, 19551.0, -24751.0...","[2199.0, -1407.0, 3097.0, -5922.0, -9417.0, -4...",[-13743.0],"[12177.0, 4314.0, 5623.0, -6518.0, -3149.0, 15...","[30931.0, 3959.0, 3501.0, 17430.0, 4641.0